# Truestar in five minutes, on a CPU

This notebook runs `truestar check` on a small model with a planted backend bug, reads the report,
runs the repro script Truestar writes, and then reads the cross chip fingerprint from two GPUs.

Everything here runs on a CPU with stock PyTorch. No GPU, CUDA toolkit or compiler is needed.

Install first, in the environment this notebook runs in:

```
pip install torch       # the build pytorch.org gives for your machine
pip install truestar    # not on PyPI yet: from a checkout, run `pip install .`
```

In [1]:
import json, os, subprocess, sys, tempfile, time
from pathlib import Path

import torch
import truestar

T0 = time.perf_counter()
NB_DIR = Path.cwd()                                   # the folder this notebook was started from
WORK = Path(tempfile.mkdtemp(prefix="truestar_demo_"))  # model file and repros go here
os.chdir(WORK)
print("truestar", truestar.__version__, "| torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())


def tidy(text):
    # Shorten machine specific paths so the output reads the same on every machine.
    for long, short in ((str(WORK) + os.sep, ""), (str(WORK), "."), (sys.executable, "python"),
                        (sys.prefix, "<python>")):
        text = text.replace(long, short)
    return text


def sh(*args, env=None, show=True):
    # Run `python <args>` in the work folder and print its output and exit code.
    t = time.perf_counter()
    p = subprocess.run([sys.executable, *args], cwd=WORK, capture_output=True, text=True,
                       env={**os.environ, **(env or {})})
    if show:
        print(tidy((p.stdout + p.stderr).rstrip()))
        print(f"[exit code {p.returncode}, {time.perf_counter() - t:.1f} s]")
    return p

truestar 0.1.0 | torch 2.11.0+cpu | CUDA available: False


## 1. A model with a planted backend bug

The model code below is clean: a 1D convolution, two residual blocks (layer norm, linear, GELU,
linear) and a classifier head. The bug is planted one level down, in the CPU kernel that PyTorch
runs for `aten::addmm` (the matrix multiply plus bias behind `nn.Linear`). Importing the file
registers a replacement kernel that adds the bias twice, but only for float32. The float64 path of
the same kernel is correct.

That is the shape of a real backend bug: the Python code is the same on every backend, and one
kernel on one backend at one dtype is wrong. Set `PLANTED_BUG=0` to load the model without it.

In [2]:
%%writefile planted_model.py
import os
import warnings

import torch
from torch import nn

_lib = None


def install_broken_addmm():
    """Replace the CPU kernel of aten::addmm with one that is wrong in float32 only."""
    global _lib
    if _lib is not None:
        return
    _lib = torch.library.Library("aten", "IMPL")

    def addmm(bias, mat1, mat2, *, beta=1, alpha=1):
        out = torch.ops.aten.mm.default(mat1, mat2)
        if alpha != 1:
            out = out * alpha
        b = bias if beta == 1 else bias * beta
        out = out + b
        if out.dtype == torch.float32:
            out = out + b                    # the planted bug: float32 adds the bias twice
        return out

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")      # PyTorch warns when a built in kernel is replaced
        _lib.impl("addmm", addmm, "CPU")


if os.environ.get("PLANTED_BUG", "1") == "1":
    install_broken_addmm()


class Block(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.norm = nn.LayerNorm(d)
        self.up = nn.Linear(d, 4 * d)
        self.down = nn.Linear(4 * d, d)

    def forward(self, x):
        return x + self.down(torch.nn.functional.gelu(self.up(self.norm(x))))


class TinyClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Conv1d(16, 64, 3, padding=1)
        self.blocks = nn.Sequential(Block(), Block())
        self.head = nn.Linear(64, 10)

    def forward(self, x):
        h = self.embed(x.transpose(1, 2)).transpose(1, 2)
        return self.head(self.blocks(h).mean(1))


# The three functions a Truestar model file defines (get_kwargs is optional).
def get_model():
    torch.manual_seed(0)
    return TinyClassifier().eval()


def get_inputs():
    torch.manual_seed(1)
    return torch.randn(8, 12, 16)

Writing planted_model.py


## 2. Run `truestar check`

On a CPU only machine the backend under test is CPU float32, and the reference is the same model
run once in float64 on the CPU. Every module is then replayed alone on the backend with the
reference inputs, so the error each module shows is its own, never inherited from upstream. Inside
the first module over its threshold, the ATen ops are traced on both sides and the first op over
its own threshold is named. On a GPU machine, add `--backends cpu,cuda` (or `mps`).

In [3]:
check = sh("-m", "truestar", "check", "planted_model.py", "--backends", "cpu")

truestar check: planted_model (float32), reference CPU float64

[DIVERGENT] cpu: CPU  TS010 TS013
  precision: linear float32 (24 bits, subnormals kept); bmm float32 (24 bits, subnormals kept); conv2d float32 (24 bits, subnormals kept)
  final output: relative error 7.1e-01
  TS013 decisions: 3/8 predictions changed (0 near ties, 3 real)
  TS010 first divergent module: blocks.0.up#0 (Linear), own error 1.2e-01 vs threshold 1.0e-04
  source: planted_model.py:36 (planted_model.Block)
  first divergent op: aten.addmm.default (op 3 of 4 in blocks.0.up#0), own error 1.2e-01 vs threshold 1.0e-04
    inputs: float32[256], float32[96, 64], float32[64, 256]
  repro: python truestar_repro/cpu_blocks.0.up_0/repro.py
  op repro: python truestar_repro/cpu_blocks.0.up_0/op_repro.py
  modules over threshold: 5 of 8 checked
    DIVERGENT blocks.0.up#0                            own  1.2e-01  accumulated  1.2e-01  threshold 1.0e-04
    DIVERGENT blocks.0.down#0                          own  1.7e-01  ac

How to read it:

* `DIVERGENT`, codes `TS010` (a module over its threshold) and `TS013` (predictions changed: some
  of the 8 inputs get a different class).
* The first divergent module is `blocks.0.up`, an `nn.Linear`. Its class lives in PyTorch, so the
  report also names the nearest class in your code, `Block` in `planted_model.py`.
* The first divergent op inside it is `aten.addmm.default`, the kernel we broke.
* Every later Linear is listed too, each with its own error, because each one calls the same broken
  kernel. Isolated replay is what keeps the GELU and the layer norms off that list.
* Exit code 1 means a finding.

The same model with the planted kernel switched off reports clean, with exit code 0:

In [4]:
clean = sh("-m", "truestar", "check", "planted_model.py", "--backends", "cpu", env={"PLANTED_BUG": "0"})

truestar check: planted_model (float32), reference CPU float64

[OK] cpu: CPU
  precision: linear float32 (24 bits, subnormals kept); bmm float32 (24 bits, subnormals kept); conv2d float32 (24 bits, subnormals kept)
  final output: relative error 1.9e-07
  decisions: 0/8 predictions changed
  all 11 checked modules within threshold (closest: blocks.0.down#0, 2.0e-07 vs 1.0e-04)

overall: OK
[exit code 0, 1.2 s]


## 3. The same result as JSON, for scripts and coding agents

`--format agent` prints one versioned JSON document with stable codes, the blamed file and line,
and a list of next actions with shell commands.

In [5]:
p = sh("-m", "truestar", "check", "planted_model.py", "--backends", "cpu", "--format", "agent", show=False)
doc = json.loads(p.stdout)
b = doc["backends"][0]
mod, op = b["blamed_module"], b["blamed_op"]
print("status:        ", doc["status"], "| exit code", doc["exit_code"], "| codes", doc["codes"])
print("blamed module: ", mod["name"], f"({mod['type']})", "own error %.1e vs threshold %.0e" % (mod["rel_err"], mod["threshold"]))
print("in your code:  ", tidy(mod["user_source"]["file"]) + ":" + str(mod["user_source"]["line"]), mod["user_source"]["qualname"])
print("blamed op:     ", op["op"], f"(op {op['index']} of {op['total']})", [f"{i['dtype']}{i['shape']}" for i in op["inputs"]])
print("next actions:  ", [a["kind"] for a in doc["next_actions"]])

status:         divergent | exit code 1 | codes ['TS010', 'TS013']
blamed module:  blocks.0.up (Linear) own error 1.2e-01 vs threshold 1e-04
in your code:   planted_model.py:36 planted_model.Block
blamed op:      aten.addmm.default (op 3 of 4) ['float32[256]', 'float32[96, 64]', 'float32[64, 256]']
next actions:   ['confirm_op_repro', 'confirm_repro', 'control_repro', 'inspect_source', 'pin_torch', 'report_upstream', 'check_flips']


## 4. The repro script

Truestar wrote a folder with `repro.py` (the blamed module alone, its weights and the reference
inputs) and `op_repro.py` (the single ATen op with its inputs). Both need only PyTorch. A repro
exits 1 when the divergence reproduces and 0 when the output matches the float64 reference.

In [6]:
repro_dir = next((WORK / "truestar_repro").iterdir())
print(tidy(str(repro_dir)), sorted(f.name for f in repro_dir.iterdir()))
print()
print("".join((repro_dir / "repro.py").read_text().splitlines(keepends=True)[:14]))

truestar_repro/cpu_blocks.0.up_0 ['op_payload.pt', 'op_repro.py', 'payload.pt', 'repro.py']

"""Repro generated by truestar check.

Layer:    blocks.0.up#0 (Linear) from model planted_model
Backend:  cpu (CPU)
Expected: relative error below 1.0e-04 versus a float64 CPU reference.

Run:  python repro.py            (uses the backend above)
      python repro.py cpu        (control run, should pass)
Needs only PyTorch. If the layer is a custom class, its module must be importable.
Exit code 1 means the divergence reproduced.
"""
import contextlib
import os
import sys



With a real backend bug the repro reproduces as is, because the broken kernel ships with the
backend. Here the broken kernel lives in `planted_model.py`, so the repro reproduces only when that
file is imported first. Run it both ways: with the planted kernel loaded it exits 1, without it
(stock PyTorch, the control) it exits 0.

In [7]:
repro = tidy(str(repro_dir / "repro.py"))
op_repro = tidy(str(repro_dir / "op_repro.py"))
launch = "import planted_model, runpy, sys; sys.argv = [{0!r}]; runpy.run_path({0!r}, run_name='__main__')"

print("--- repro.py with the planted kernel loaded")
r1 = sh("-c", launch.format(repro))
print("--- op_repro.py with the planted kernel loaded")
r2 = sh("-c", launch.format(op_repro))
print("--- repro.py on stock PyTorch (control)")
r3 = sh(repro)
assert (r1.returncode, r2.returncode, r3.returncode) == (1, 1, 0)

--- repro.py with the planted kernel loaded


torch 2.11.0+cpu on cpu: relative error 1.197e-01 (threshold 1.0e-04)
REPRODUCED: output diverges from the float64 CPU reference
[exit code 1, 0.6 s]
--- op_repro.py with the planted kernel loaded


torch 2.11.0+cpu aten.addmm.default on cpu: relative error 1.197e-01 (threshold 1.0e-04)
[exit code 1, 0.6 s]
--- repro.py on stock PyTorch (control)


torch 2.11.0+cpu on cpu: relative error 1.486e-07 (threshold 1.0e-04)
ok: output matches the reference
[exit code 0, 0.6 s]


## 5. The cross chip fingerprint

`scripts/laptop_fingerprint.py` builds three small models from fixed seeds (an MLP, a
`TransformerEncoderLayer` and resnet18 at 64 x 64), hashes their weights and inputs, and records a
SHA256 of the raw output bytes for eager PyTorch on CUDA and for Truestar's compiled executor, each
run twice. Equal hashes mean bit for bit equal outputs. Settings: float32, TF32 off,
`torch.use_deterministic_algorithms(True)`, batch 4, and the compiled executor in its deterministic
mode (no autotuning, no split K).

It was run on two machines. The two JSON files ship in `docs/data/cross_chip/`.

In [8]:
names = ("fp_desktop_5070ti.json", "fp_laptop_5060.json")
data_dir = next((d for d in (NB_DIR / "../../docs/data/cross_chip", NB_DIR / "docs/data/cross_chip")
                 if (d / names[0]).exists()), None)
if data_dir is not None:
    fps = [json.loads((data_dir / n).read_text()) for n in names]
else:                                                 # notebook opened outside a checkout
    import urllib.request
    base = "https://raw.githubusercontent.com/SharvenRane/truestar/main/docs/data/cross_chip/"
    fps = [json.loads(urllib.request.urlopen(base + n).read()) for n in names]
a, b = fps

keys = ("gpu", "sm_count", "capability", "driver_cuda", "torch", "cudnn", "triton")
print(f"{'':12} {'machine A':36} {'machine B':36}")
for k in keys:
    print(f"{k:12} {str(a['env'][k]):36} {str(b['env'][k]):36}")

             machine A                            machine B                           
gpu          NVIDIA GeForce RTX 5070 Ti           NVIDIA GeForce RTX 5060 Laptop GPU  
sm_count     70                                   26                                  
capability   12.0                                 12.0                                
driver_cuda  12.8                                 12.8                                
torch        2.11.0+cu128                         2.11.0+cu128                        
cudnn        91900                                91900                               
triton       3.7.1                                3.7.1                               


In [9]:
def same(x, y):
    return "same" if x == y else "DIFFERENT"

print(f"{'case':12} {'eager CUDA A':14} {'eager CUDA B':14} {'A vs B':10} {'compiled A':14} {'compiled B':14} {'A vs B':10} {'max err vs float64':>18}")
for ra, rb in zip(a["results"], b["results"]):
    assert ra["weights_sha256"] == rb["weights_sha256"] and ra["input_sha256"] == rb["input_sha256"]
    ea, eb = ra["eager_cuda"][0], rb["eager_cuda"][0]
    ca, cb = ra["compiled_cuda"][0], rb["compiled_cuda"][0]
    print(f"{ra['case']:12} {ea[:12]:14} {eb[:12]:14} {same(ea, eb):10} {ca[:12]:14} {cb[:12]:14} {same(ca, cb):10} {ra['compiled_max_error_vs_float64']:18.1e}")

print()
for r in a["results"] + b["results"]:
    assert r["compiled_run_to_run_equal"] and r["eager_run_to_run_equal"]
print("run to run: every eager and compiled output hashed the same on its second call, on both machines")
print("compiled equals eager CUDA bit for bit:", sorted({r["compiled_equals_eager_cuda_bitwise"] for r in a["results"] + b["results"]}))

case         eager CUDA A   eager CUDA B   A vs B     compiled A     compiled B     A vs B     max err vs float64
mlp          7e6834dbcdcd   7e6834dbcdcd   same       0045ade06d6e   0045ade06d6e   same                  5.8e-07
transformer  bb3a9fca1d37   7cc508054f96   DIFFERENT  36adb88d5d72   36adb88d5d72   same                  1.9e-07
resnet18     3d786df3d6ac   3d786df3d6ac   same       c83acd13165e   c83acd13165e   same                  3.0e-06

run to run: every eager and compiled output hashed the same on its second call, on both machines
compiled equals eager CUDA bit for bit: [False]


What this shows, and what it does not:

> On two NVIDIA consumer Blackwell GPUs (RTX 5070 Ti, 70 SMs, and RTX 5060 Laptop, 26 SMs), with
> the same driver, CUDA 12.8, PyTorch 2.11, Triton 3.7.1 and cuDNN 9.19, Truestar's compiled float32
> outputs for a small MLP, a TransformerEncoderLayer and resnet18 were bitwise identical run to run
> and across the two GPUs; eager PyTorch differed between the GPUs on the transformer layer. This
> was measured at batch 4 with TF32 off and the compiled executor in deterministic mode (no
> autotuning, no split K). We have not tested other GPU architectures, AMD or Apple GPUs, other
> Triton or driver versions, half precision, TF32, or tuned configs, and we expect differences
> there. Truestar does not claim bitwise equality with PyTorch or with float64; its compiled results
> are checked against float64 within a stated bound.

Two more details from the same files. Eager PyTorch matched across the two GPUs on the MLP and
resnet18, so the contrast is one case of three. resnet18's convolutions run as cuDNN library calls
inside the compiled executor, so their agreement is cuDNN choosing the same algorithm on both GPUs,
not something Truestar guarantees.

On a machine with an NVIDIA GPU you can add your own row: run the script below from a checkout and
compare your JSON with the two above.

In [10]:
if torch.cuda.is_available():
    print("From a checkout:  PYTHONPATH=src python scripts/laptop_fingerprint.py --json my_gpu.json")
else:
    print("No CUDA GPU here, so there is no fingerprint to add from this machine.")
print(f"notebook time: {time.perf_counter() - T0:.1f} s")

No CUDA GPU here, so there is no fingerprint to add from this machine.
notebook time: 5.4 s
